# Per-garden GWAS on the ecotype selection coefficient — GEMMA results

30 gardens × 3 marker classes, each an **independent** GEMMA v0.98.5 univariate LMM
(`-lmm 1`, Wald) on the rank-inverse-normalized per-founder selection coefficient, with a
leave-one-chromosome-out GRM built from all classes pooled.

There is **no cross-site meta** — the all-sites question belongs to the GEA track. So a marker
significant in one garden is a per-garden result, and cross-garden recurrence below is
**descriptive, not a tested contrast**.

Methods of record: `r3_persite_gwas/METHODS_PERSITE_GWAS.md`.


In [1]:

import os, json
import numpy as np, pandas as pd
from scipy import stats
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "font.size": 8, "axes.grid": False})

RES = "/global/scratch/projects/fc_moilab/tbellg/kmate/analysis/grenenet_selection/r3_persite_gwas/results/gemma_gwas"
PLOTS = f"{RES}/plots"; os.makedirs(PLOTS, exist_ok=True)
CLASSES = ["snp", "nonsnp", "sv"]
COL = {"snp": "#4C72B0", "nonsnp": "#DD8452", "sv": "#55A868"}

D = {c: np.load(f"{RES}/persite_gwas_{c}.npz", allow_pickle=True) for c in CLASSES}
summary = json.load(open(f"{RES}/persite_gwas_summary.json"))
gardens = np.array(summary["gardens"]); bio1 = np.array(summary["bio1"])
order = np.argsort(bio1)          # cold -> hot, the standard ordering in this project
print(f"{len(gardens)} gardens, bio1 {bio1.min():.1f}-{bio1.max():.1f} C")
for c in CLASSES:
    print(f"  {c:7s} {D[c]['Z'].shape[0]:>9,d} markers x {D[c]['Z'].shape[1]} gardens")


30 gardens, bio1 6.0-21.6 C


  snp     1,752,846 markers x 30 gardens


  nonsnp    525,043 markers x 30 gardens
  sv         12,789 markers x 30 gardens


## 1. Per-garden λ and Bonferroni hit counts

In [2]:

rows = []
for c in CLASSES:
    P = D[c]["P"]; M = P.shape[0]; thr = 0.05 / M
    for si in range(len(gardens)):
        p = P[:, si]; ok = np.isfinite(p)
        rows.append(dict(cls=c, garden=int(gardens[si]), bio1=float(bio1[si]),
                         lam=float(np.median(stats.norm.ppf(1 - p[ok] / 2) ** 2) / stats.chi2.ppf(0.5, 1)),
                         n_bonf=int((p[ok] < thr).sum()),
                         best_nlp=float(-np.log10(np.nanmin(p[ok]))) if ok.any() else np.nan))
tab = pd.DataFrame(rows)
piv = tab.pivot(index="garden", columns="cls", values="n_bonf").reindex(gardens[order])
piv["bio1"] = bio1[order].round(1)
lam = tab.pivot(index="garden", columns="cls", values="lam").reindex(gardens[order]).round(3)
print("Bonferroni hits per garden (cold -> hot):"); print(piv.to_string())
print(f"\nTotal garden-hits: {tab.groupby('cls').n_bonf.sum().to_dict()}")
uniq = {c: int(((D[c]["P"] < 0.05 / D[c]["P"].shape[0]).any(1)).sum()) for c in CLASSES}
print(f"UNIQUE markers hit in >=1 garden: {uniq}")
print(f"\nlambda per garden -- median {lam.median().round(3).to_dict()}")


Bonferroni hits per garden (cold -> hot):
cls     nonsnp  snp  sv  bio1
garden                       
24           0    0   1   6.0
23           0    0   0   7.3
27          17   23   6   7.4
37           0    1   1   8.1
48           1    0   0   8.5
1            1    0   1  10.7
42          10   13   2  10.9
46           0    0   0  11.3
25           4    1   2  11.4
55          31   47   6  11.6
49           0    2   1  11.7
52           2    1   0  12.0
57           0    0   0  12.3
53           0    0   0  12.4
54           4    1   1  12.4
9            2    1   1  12.5
12          18   31   6  13.0
11           5   13   2  13.8
6            0    0   0  15.0
5            0    0   0  15.0
2            0    0   0  15.6
45           0    0   1  15.7
28           3    1   0  16.9
4            0    0   0  17.5
32           0    0   0  17.8
43           0    0   0  18.0
60           9   21   2  19.2
13           2    1   1  20.4
10           4    9   2  20.6
26           0    0   1  21.

UNIQUE markers hit in >=1 garden: {'snp': 112, 'nonsnp': 71, 'sv': 16}

lambda per garden -- median {'nonsnp': 1.015, 'snp': 1.026, 'sv': 1.008}


## 2. QQ per garden — 30 panels, ordered cold → hot

In [3]:

def qq_ax(ax, p, color, label):
    p = p[np.isfinite(p)]; p = np.sort(p)
    n = len(p)
    exp = -np.log10((np.arange(1, n + 1) - 0.5) / n)
    obs = -np.log10(np.clip(p, 1e-300, 1))
    # thin the dense null end so the figure stays light without distorting the tail
    keep = np.concatenate([np.arange(0, min(n, 5000)),
                           np.arange(5000, n, max(1, n // 3000))]) if n > 8000 else np.arange(n)
    ax.plot(exp[keep], obs[keep], ".", ms=1.5, color=color, rasterized=True)
    m = max(exp.max(), obs.max())
    ax.plot([0, m], [0, m], "-", lw=0.6, color="0.5")
    lamv = np.median(stats.norm.ppf(1 - p / 2) ** 2) / stats.chi2.ppf(0.5, 1)
    ax.annotate(f"{label}\n$\\lambda$={lamv:.2f}", xy=(0.04, 0.96), xycoords="axes fraction",
                va="top", ha="left", fontsize=6.5)
    ax.set_xticks([]); ax.set_yticks([])

for c in CLASSES:
    P = D[c]["P"]
    fig, axes = plt.subplots(5, 6, figsize=(13, 10.5), sharex=True, sharey=True)
    for k, si in enumerate(order):
        qq_ax(axes.flat[k], P[:, si], COL[c], f"g{gardens[si]} ({bio1[si]:.0f}C)")
    for ax in axes.flat[len(order):]:
        ax.axis("off")
    fig.supxlabel("expected $-\\log_{10}p$"); fig.supylabel("observed $-\\log_{10}p$")
    fig.tight_layout()
    fig.savefig(f"{PLOTS}/qq_grid_{c}.png", dpi=140, bbox_inches="tight")
    print(f"wrote qq_grid_{c}.png  (30 gardens, cold->hot)")
    plt.close(fig)


wrote qq_grid_snp.png  (30 gardens, cold->hot)


wrote qq_grid_nonsnp.png  (30 gardens, cold->hot)


wrote qq_grid_sv.png  (30 gardens, cold->hot)


## ⚠️ Read the MAC-stratified QQ before interpreting any hit

Genomic-control λ is ~1.0 in every garden and every MAC stratum, which looks like clean
calibration. It is not sufficient: λ is a **median** statistic and never touches the extreme
quantiles that Bonferroni hits are drawn from. Stratified by MAC, the tail runs **5–60×** hot,
and **90% of significant SNP markers sit in MAC 5–11**, the worst-calibrated stratum.

Not all artifact — a 3–4× excess at p<1e-4 persists in clean strata (expected for a polygenic
trait plus LD), and the inflation is non-monotone, rising again at MAC ≥46 where power says real
signal belongs. Rare-end artifact superimposed on common-end signal; these numbers cannot
separate them. **Open decision: permutation null per stratum, or raise the MAC floor.**


## 3. QQ by MAC stratum — the decision panel

In [4]:

# THE decision panel: QQ split by MAC stratum. lambda is ~1.0 in every stratum while the tail
# runs 5-60x hot -- lambda is a median statistic and never touches the quantiles hits live in.
mc = pd.read_csv(f"{RES}/mac_calibration.csv")
mac = {c: np.load(f"{RES}/mac_by_class.npz")[c] for c in CLASSES}
BINS = [(5, 7), (8, 11), (12, 22), (23, 45), (46, 10**9)]
fig, axes = plt.subplots(1, 3, figsize=(13, 4.4))
for ax, c in zip(axes, CLASSES):
    P = D[c]["P"]; m_all = mac[c]
    for (lo, hi), shade in zip(BINS, np.linspace(0.15, 0.85, len(BINS))):
        sel = (m_all >= lo) & (m_all <= hi)
        if sel.sum() < 100:
            continue
        p = P[sel].ravel(); p = p[np.isfinite(p)]
        n = len(p); p = np.sort(p)
        exp = -np.log10((np.arange(1, n + 1) - 0.5) / n)
        obs = -np.log10(np.clip(p, 1e-300, 1))
        keep = np.concatenate([np.arange(0, min(n, 4000)),
                               np.arange(4000, n, max(1, n // 2500))]) if n > 6000 else np.arange(n)
        lab = f"MAC {lo}-{hi}" if hi < 10**9 else f"MAC$\\geq${lo}"
        ax.plot(exp[keep], obs[keep], ".", ms=1.5, rasterized=True,
                color=plt.cm.viridis(shade), label=lab)
    m = 7.5
    ax.plot([0, m], [0, m], "-", lw=0.7, color="0.4")
    ax.annotate(c, xy=(0.04, 0.96), xycoords="axes fraction", va="top", fontsize=9, weight="bold")
    ax.set_xlabel("expected $-\\log_{10}p$"); ax.set_xlim(0, m)
    ax.legend(fontsize=6, loc="lower right", markerscale=4, frameon=False)
axes[0].set_ylabel("observed $-\\log_{10}p$")
fig.tight_layout(); fig.savefig(f"{PLOTS}/qq_by_mac_stratum.png", dpi=150, bbox_inches="tight")
print("wrote qq_by_mac_stratum.png")
plt.close(fig)
print(mc[["cls", "mac_lo", "mac_hi", "lambda_gc", "ratio_1e4", "ratio_1e6"]].to_string(index=False))


wrote qq_by_mac_stratum.png
   cls  mac_lo  mac_hi  lambda_gc  ratio_1e4  ratio_1e6
   snp       5       7   0.936080   4.669942  59.506182
   snp       8      11   1.011216   4.036259  24.566470
   snp      12      22   1.029891   2.863553  10.201846
   snp      23      45   1.062270   3.754768   5.611609
   snp      46     115   1.084347   3.903451  18.937452
nonsnp       5       7   0.940011   5.197216  51.146199
nonsnp       8      11   1.002364   4.010166  30.909184
nonsnp      12      22   1.014210   2.862124  11.378473
nonsnp      23      45   1.042522   3.297091   8.226274
nonsnp      46     115   1.074611   3.554935  21.944045
    sv       5       7   0.964816   5.884544  52.141527
    sv       8      11   0.988820   4.399586  25.879917
    sv      12      22   1.030071   5.579912 119.569550
    sv      23      45   1.079449   2.388760   0.000000
    sv      46     115   1.062220   4.329090   0.000000


## 4. Manhattan per garden — 30 panels, ordered cold → hot

In [5]:

CH = ["chr1", "chr2", "chr3", "chr4", "chr5"]
def genome_coords(chrom, pos):
    off, cum = {}, 0
    for c in CH:
        off[c] = cum; cum += pos[chrom == c].max() if (chrom == c).any() else 0
    x = np.array([off[c] for c in chrom]) + pos
    centers = [off[c] + (pos[chrom == c].max() / 2 if (chrom == c).any() else 0) for c in CH]
    return x, centers, cum

for c in CLASSES:
    chrom = D[c]["chrom"].astype(str); pos = D[c]["pos"].astype(float); P = D[c]["P"]
    x, centers, tot = genome_coords(chrom, pos)
    band = np.isin(chrom, CH[::2])
    thr = -np.log10(0.05 / P.shape[0])
    fig, axes = plt.subplots(10, 3, figsize=(14, 18), sharex=True, sharey=True)
    for k, si in enumerate(order):
        ax = axes.flat[k]
        nlp = -np.log10(np.clip(P[:, si], 1e-300, 1))
        # keep every informative point, thin the dense null floor 20x -- plotting all ~550k
        # sub-threshold markers x 30 panels is 16M glyphs per figure and renders nothing extra
        strong = nlp > 2.0
        floor = np.where(nlp <= 2.0)[0][::20]
        keep = np.zeros(len(nlp), bool); keep[strong] = True; keep[floor] = True
        ax.scatter(x[keep & band], nlp[keep & band], s=0.7, c="0.62", rasterized=True, lw=0)
        ax.scatter(x[keep & ~band], nlp[keep & ~band], s=0.7, c=COL[c], rasterized=True, lw=0)
        ax.axhline(thr, color="crimson", lw=0.6, ls="--")
        nb = int((P[:, si] < 0.05 / P.shape[0]).sum())
        ax.annotate(f"g{gardens[si]}  {bio1[si]:.0f}$^\\circ$C  n={nb}", xy=(0.01, 0.94),
                    xycoords="axes fraction", va="top", fontsize=6.5)
        ax.set_xticks([]); ax.set_ylim(0.5, max(thr + 1.5, np.nanmax(nlp) * 1.05))
    for ax in axes.flat[len(order):]:
        ax.axis("off")
    fig.supylabel("$-\\log_{10}p$"); fig.supxlabel("genome position (chr1-chr5)")
    fig.tight_layout()
    fig.savefig(f"{PLOTS}/manhattan_grid_{c}.png", dpi=130, bbox_inches="tight")
    print(f"wrote manhattan_grid_{c}.png")
    plt.close(fig)


wrote manhattan_grid_snp.png


wrote manhattan_grid_nonsnp.png


wrote manhattan_grid_sv.png


## 5. The hit list, with MAC attached

In [6]:

# the actual hit list, with MAC attached -- the column that decides whether to trust a row
recs = []
for c in CLASSES:
    P = D[c]["P"]; thr = 0.05 / P.shape[0]
    chrom = D[c]["chrom"].astype(str); pos = D[c]["pos"]
    m_all = mac[c]
    hit_i, hit_g = np.where(P < thr)
    for i, g in zip(hit_i, hit_g):
        recs.append(dict(cls=c, chrom=chrom[i], pos=int(pos[i]), garden=int(gardens[g]),
                         bio1=round(float(bio1[g]), 1), mac=int(m_all[i]),
                         maf=round(m_all[i] / 231, 4),
                         nlp=round(float(-np.log10(max(P[i, g], 1e-300))), 2)))
hits = pd.DataFrame(recs).sort_values("nlp", ascending=False)
hits.to_csv(f"{RES}/persite_bonferroni_hits.csv", index=False)
print(f"{len(hits)} garden-hits; {hits.groupby('cls').size().to_dict()}")
print(f"\nMAC distribution of hits vs panel:")
for c in CLASSES:
    h = hits[hits.cls == c]
    print(f"  {c:7s} hits MAC<12: {(h.mac<12).mean():5.1%}  |  panel MAC<12: {(mac[c]<12).mean():5.1%}")
print("\nTop 15 by -log10 p:")
print(hits.head(15).to_string(index=False))


316 garden-hits; {'nonsnp': 113, 'snp': 166, 'sv': 37}

MAC distribution of hits vs panel:
  snp     hits MAC<12: 91.6%  |  panel MAC<12: 36.6%
  nonsnp  hits MAC<12: 79.6%  |  panel MAC<12: 35.8%
  sv      hits MAC<12: 54.1%  |  panel MAC<12: 55.1%

Top 15 by -log10 p:
   cls chrom      pos  garden  bio1  mac    maf   nlp
   snp  chr2 13900692      27   7.4    8 0.0346 10.83
   snp  chr4 18399380      55  11.6    5 0.0216 10.68
nonsnp  chr5   482054      12  13.0    5 0.0216 10.10
   snp  chr5 16190357      12  13.0    5 0.0216  9.98
   snp  chr5 16190666      12  13.0    5 0.0216  9.88
   snp  chr4  1254136      55  11.6    5 0.0216  9.66
nonsnp  chr2 16378782      12  13.0    5 0.0216  9.51
   snp  chr4 15204084      55  11.6    5 0.0216  9.29
   snp  chr5 19620565      55  11.6    5 0.0216  9.27
   snp  chr4 12170010      12  13.0   10 0.0433  9.18
   snp  chr5  3161591      42  10.9    5 0.0216  9.12
   snp  chr5  3163105      42  10.9    5 0.0216  9.12
   snp  chr1  4707836      

## 6. Significant blocks per garden, and overlap across gardens

Block unit = the **clq0.9 tiling partition, unmerged**, identical for all three classes.
Two deliberate departures from the retired notebook:

- **tiling, not the BigLD LD-islands** (`lib.assign_clq_blocks`) — strict containment silently
  drops ~28% of variants into inter-block gaps. The old table used the island version.
- **no `merge_small_blocks`** — it re-partitions from whichever records it is given, so merging
  per class would make an SV block a far larger genomic unit than a SNP block and the counts
  would not be comparable. Unmerged, a block is the same interval for every class.

Cross-garden recurrence below is **descriptive, not a tested contrast** — that test was the
meta this analysis dropped, and the 30 gardens share one founder panel and one `p0`, so their
scans are correlated by construction.

In [7]:

import sys
sys.path.insert(0, "/global/scratch/projects/fc_moilab/tbellg/kmate/analysis/grenenet_selection"); sys.path.insert(0, "/global/scratch/projects/fc_moilab/tbellg/kmate/analysis/grenenet_selection/blocks")
import blocks_tiling as bt

# Block unit = the clq0.9 TILING partition, UNMERGED and identical for all three classes.
# Two deliberate choices:
#  * tiling, not lib.assign_clq_blocks (the BigLD LD-islands): strict containment silently
#    drops ~28% of variants into inter-block gaps. The old notebook used the island version.
#  * NO merge_small_blocks: it re-partitions from whichever records you hand it, so merging
#    per class would make an SV "block" a much larger genomic unit than a SNP block and the
#    counts would not be comparable. Unmerged, a block is the same interval for every class.
BLK = {}
for c in CLASSES:
    ch = np.char.replace(D[c]["chrom"].astype(str), "chr", "Chr")
    BLK[c] = bt.assign_tiling(ch, D[c]["pos"], r2=0.9)
    print(f"{c:7s} {len(BLK[c]):>9,d} markers -> {len(set(BLK[c])):>6,d} blocks carrying markers")

sig_blocks = {}          # (class, garden) -> set of blocks with >=1 Bonferroni marker
for c in CLASSES:
    P = D[c]["P"]; thr = 0.05 / P.shape[0]; b = np.asarray(BLK[c], dtype=object)
    for si in range(len(gardens)):
        hit = np.isfinite(P[:, si]) & (P[:, si] < thr)
        sig_blocks[(c, int(gardens[si]))] = set(b[hit])


snp     1,752,846 markers -> 58,247 blocks carrying markers


nonsnp    525,043 markers -> 51,024 blocks carrying markers
sv         12,789 markers ->  7,057 blocks carrying markers


In [8]:

rows = []
for si in order:
    g = int(gardens[si])
    r = {"garden": g, "bio1": round(float(bio1[si]), 1)}
    for c in CLASSES:
        r[c] = len(sig_blocks[(c, g)])
    r["any_class"] = len(set().union(*[sig_blocks[(c, g)] for c in CLASSES]))
    rows.append(r)
bt_tab = pd.DataFrame(rows)
bt_tab.loc["TOTAL_distinct"] = ["-", "-"] + [
    len(set().union(*[sig_blocks[(c, int(g))] for g in gardens])) for c in CLASSES] + [
    len(set().union(*[sig_blocks[(c, int(g))] for c in CLASSES for g in gardens]))]
print("Bonferroni-significant BLOCKS per garden (clq0.9 tiling, cold -> hot)\n")
print(bt_tab.to_string(index=False))
bt_tab.to_csv(f"{RES}/persite_sig_blocks_per_garden.csv", index=False)
print(f"\nGardens with >=1 significant block: " +
      ", ".join(f"{c}={sum(1 for g in gardens if sig_blocks[(c,int(g))])}/{len(gardens)}" for c in CLASSES))


Bonferroni-significant BLOCKS per garden (clq0.9 tiling, cold -> hot)

garden  bio1  snp  nonsnp  sv  any_class
    24   6.0    0       0   1          1
    23   7.3    0       0   0          0
    27   7.4   20      17   6         41
    37   8.1    1       0   1          2
    48   8.5    0       1   0          1
     1  10.7    0       1   1          2
    42  10.9    9       9   2         19
    46  11.3    0       0   0          0
    25  11.4    1       4   2          7
    55  11.6   36      22   6         55
    49  11.7    1       0   1          2
    52  12.0    1       1   0          2
    57  12.3    0       0   0          0
    53  12.4    0       0   0          0
    54  12.4    1       2   1          3
     9  12.5    1       2   1          4
    12  13.0   28      18   6         46
    11  13.8   13       5   2         19
     6  15.0    0       0   0          0
     5  15.0    0       0   0          0
     2  15.6    0       0   0          0
    45  15.7    0       0  

In [9]:

# ---- overlap across gardens: how often does the SAME block recur? ----
# DESCRIPTIVE ONLY. There is no cross-garden test here -- that was the meta this analysis
# deliberately dropped. Recurrence is also inflated by the fact that the 30 gardens share one
# founder panel and one p0, so their scans are correlated by construction.
print("Recurrence of significant blocks across gardens (descriptive, NOT a tested contrast)\n")
rec_rows = []
for c in CLASSES:
    from collections import Counter
    cnt = Counter()
    for g in gardens:
        for b in sig_blocks[(c, int(g))]:
            cnt[b] += 1
    dist = Counter(cnt.values())
    tot = len(cnt)
    rec_rows.append(dict(cls=c, blocks_total=tot,
                         in_1=dist.get(1, 0),
                         in_2=dist.get(2, 0),
                         in_3plus=sum(v for k, v in dist.items() if k >= 3),
                         max_gardens=max(cnt.values()) if cnt else 0))
    top = cnt.most_common(5)
    print(f"  {c:7s} {tot:4d} distinct blocks | recurring in >=2 gardens: "
          f"{sum(1 for v in cnt.values() if v >= 2):3d} | top: {top}")
rec = pd.DataFrame(rec_rows)
print("\n" + rec.to_string(index=False))
rec.to_csv(f"{RES}/persite_block_recurrence.csv", index=False)

# ---- pairwise garden overlap (Jaccard), per class ----
fig, axes = plt.subplots(1, 3, figsize=(14, 4.6))
for ax, c in zip(axes, CLASSES):
    S = [sig_blocks[(c, int(gardens[si]))] for si in order]
    n = len(S); J = np.full((n, n), np.nan)
    for i in range(n):
        for j in range(n):
            u = len(S[i] | S[j])
            J[i, j] = len(S[i] & S[j]) / u if u else np.nan
    im = ax.imshow(J, cmap="magma", vmin=0, vmax=np.nanmax(J[~np.eye(n, dtype=bool)]) or 1)
    ax.annotate(c, xy=(0.03, 0.97), xycoords="axes fraction", va="top",
                fontsize=9, weight="bold", color="w")
    ax.set_xticks(range(0, n, 5)); ax.set_xticklabels(gardens[order][::5], fontsize=6)
    ax.set_yticks(range(0, n, 5)); ax.set_yticklabels(gardens[order][::5], fontsize=6)
    ax.set_xlabel("garden (cold $\\rightarrow$ hot)")
    plt.colorbar(im, ax=ax, fraction=0.046, label="Jaccard")
axes[0].set_ylabel("garden (cold $\\rightarrow$ hot)")
fig.tight_layout(); fig.savefig(f"{PLOTS}/block_overlap_jaccard.png", dpi=150, bbox_inches="tight")
print("\nwrote block_overlap_jaccard.png")
plt.close(fig)

# ---- cross-CLASS overlap within a garden ----
print("\nWithin-garden overlap between classes (blocks flagged by snp AND by nonsnp):")
xs = []
for si in order:
    g = int(gardens[si]); a, b_ = sig_blocks[("snp", g)], sig_blocks[("nonsnp", g)]
    if a or b_:
        xs.append(dict(garden=g, bio1=round(float(bio1[si]), 1), snp=len(a), nonsnp=len(b_),
                       shared=len(a & b_), nonsnp_only=len(b_ - a)))
xdf = pd.DataFrame(xs)
print(xdf.to_string(index=False) if len(xdf) else "  (none)")
if len(xdf):
    print(f"\n  total non-SNP-only blocks across gardens: {xdf.nonsnp_only.sum()}")
    xdf.to_csv(f"{RES}/persite_block_class_overlap.csv", index=False)


Recurrence of significant blocks across gardens (descriptive, NOT a tested contrast)

  snp       79 distinct blocks | recurring in >=2 gardens:  26 | top: [('Chr4_9231', 6), ('Chr2_7040', 5), ('Chr5_9812', 5), ('Chr2_7039', 5), ('Chr1_6385', 5)]
  nonsnp    57 distinct blocks | recurring in >=2 gardens:  17 | top: [('Chr5_3057', 6), ('Chr1_6385', 6), ('Chr3_6055', 4), ('Chr3_3955', 4), ('Chr5_11702', 4)]
  sv        16 distinct blocks | recurring in >=2 gardens:   7 | top: [('Chr5_835', 13), ('Chr1_4693', 3), ('Chr3_5089', 3), ('Chr2_1996', 3), ('Chr1_7295', 2)]

   cls  blocks_total  in_1  in_2  in_3plus  max_gardens
   snp            79    53    14        12            6
nonsnp            57    40     7        10            6
    sv            16     9     3         4           13



wrote block_overlap_jaccard.png

Within-garden overlap between classes (blocks flagged by snp AND by nonsnp):
 garden  bio1  snp  nonsnp  shared  nonsnp_only
     27   7.4   20      17       1           16
     37   8.1    1       0       0            0
     48   8.5    0       1       0            1
      1  10.7    0       1       0            1
     42  10.9    9       9       0            9
     25  11.4    1       4       0            4
     55  11.6   36      22       6           16
     49  11.7    1       0       0            0
     52  12.0    1       1       0            1
     54  12.4    1       2       1            1
      9  12.5    1       2       0            2
     12  13.0   28      18       4           14
     11  13.8   13       5       1            4
     28  16.9    1       2       1            1
     60  19.2    8       3       2            1
     13  20.4    1       2       0            2
     10  20.6    8       4       1            3

  total non-SNP-only blo